<a href="https://colab.research.google.com/github/Rahma-Nafisa/Scraping-data-dari-aplikasi-Duolingo/blob/main/Analisis_Sentiment_Duolingo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-play-scraper
!pip install snscrape

from google_play_scraper import app, reviews, Sort, reviews_all

import pandas as pd
pd.options.mode.chained_assignment = None
import numpy as np
seed = 0
np.random.seed(seed)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score


import datetime as dt
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from wordcloud import WordCloud

import nltk
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
import pandas as pd


df = pd.read_csv('ulasan_aplikasi.csv')

print(df.columns)

In [ ]:
import pandas as pd
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

df = pd.read_csv('ulasan_aplikasi.csv')
df = df[['content']].dropna()


def cleaningText(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text_clean'] = df['content'].apply(cleaningText)


slangwords = {
    "gk": "tidak", "ga": "tidak", "nggak": "tidak", "tdk": "tidak",
    "bgt": "banget", "dr": "dari", "dgn": "dengan", "tp": "tapi",
    "udh": "sudah", "blm": "belum", "trs": "terus", "jg": "juga"
}

def fix_slangwords(text):
    return ' '.join([slangwords.get(word, word) for word in text.split()])

df['text_slang'] = df['text_clean'].apply(fix_slangwords)


def tokenizingText(text):
    return text.split()

df['text_tokens'] = df['text_slang'].apply(tokenizingText)


stop_words = set(stopwords.words('indonesian'))

def filteringText(words):
    return [word for word in words if word not in stop_words]

df['text_stopwords'] = df['text_tokens'].apply(filteringText)


def toSentence(words):
    return ' '.join(words)

df['text_final'] = df['text_stopwords'].apply(toSentence)


print(df[['content', 'text_final']].head())

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemmingText(text):
    return stemmer.stem(text)


df['text_stemmed'] = df['text_final'].apply(stemmingText)


print(df[['content', 'text_final', 'text_stemmed']].head())

In [ ]:
positive_words = {
    'bagus', 'baik', 'mantap', 'suka', 'cepat', 'puas', 'lancar', 'mudah',
    'keren', 'mantul', 'top', 'rekomendasi', 'senang', 'cocok', 'berhasil'
}
negative_words = {
    'buruk', 'jelek', 'lama', 'error', 'parah', 'lemot', 'kecewa', 'susah',
    'gagal', 'crash', 'keluar', 'hang', 'payah', 'benci', 'ribet'
}

# Fungsi analisis sentimen
def sentiment_analysis_lexicon_indonesia(text):
    score = 0
    for word in text.split():
        if word in positive_words:
            score += 1
        elif word in negative_words:
            score -= 1
    if score > 0:
        label = 'positif'
    elif score < 0:
        label = 'negatif'
    else:
        label = 'netral'
    return score, label

In [ ]:
results = df['text_final'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))
df['polarity_score'] = results[0]
df['polarity'] = results[1]



In [ ]:
print(df[['content', 'text_stemmed', 'polarity_score', 'polarity']].head(10))

In [ ]:
X = df['text_stemmed']
y = df['polarity']

In [ ]:
from sklearn.model_selection import train_test_split

# 80/20
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 70/30
X_train_70, X_test_30, y_train_70, y_test_30 = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# 60/40
X_train_60, X_test_40, y_train_60, y_test_40 = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)

In [ ]:
print("Distribusi label skema 80/20:\n", y_train_80.value_counts(normalize=True))
print("\nDistribusi label skema 70/30:\n", y_train_70.value_counts(normalize=True))
print("\nDistribusi label skema 60/40:\n", y_train_60.value_counts(normalize=True))

In [ ]:
data_splits = {
    '80_20': (X_train_80, X_test_20, y_train_80, y_test_20),
    '70_30': (X_train_70, X_test_30, y_train_70, y_test_30),
    '60_40': (X_train_60, X_test_40, y_train_60, y_test_40)
}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

def train_and_evaluate(model, split_key, model_name='Model'):
    X_train, X_test, y_train, y_test = data_splits[split_key]

    # Pipeline TF-IDF + Model
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
        ('clf', model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    print(f"\n=== Evaluasi untuk {model_name} ({split_key}) ===")
    print(classification_report(y_test, y_pred))
    print(f"\nAkurasi: {accuracy_score(y_test, y_pred):.4f}")


    cm = confusion_matrix(y_test, y_pred, labels=['positif', 'netral', 'negatif'])
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=['positif', 'netral', 'negatif'],yticklabels=['positif', 'netral', 'negatif'], cmap="Blues")
    plt.title(f"Confusion Matrix {model_name} ({split_key})")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    return pipeline


In [ ]:
# Skema 1: Tf-IDF + SVM (80/20)
svm_model_80 = train_and_evaluate(SVC(kernel='linear'), '80_20', 'SVM')

# Skema 2: Tf-IDF + RF (70/30)
rf_model_70 = train_and_evaluate(RandomForestClassifier(n_estimators=100), '70_30', 'Random Forest')

# Skema 3: Tf-IDF + SVM (60/40)
svm_model_60 = train_and_evaluate(SVC(kernel='linear'), '60_40', 'SVM 60/40')

In [ ]:
pipeline = train_and_evaluate(SVC(), '80_20', 'SVM')

# Sample untuk prediksi
sample = ["aplikasinya bagus banget dan bermanfaat"]

prediction = pipeline.predict(sample)

print("Prediksi:", prediction)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import classification_report, accuracy_score

le = LabelEncoder()
y = le.fit_transform(df['polarity'])
num_classes = len(le.classes_)

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text_stemmed'])

sequences = tokenizer.texts_to_sequences(df['text_stemmed'])
padded_sequences = pad_sequences(sequences, padding='post', maxlen=100)

y_cat = to_categorical(y)

from sklearn.model_selection import train_test_split

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    padded_sequences, y_cat, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=128, input_length=100))
model.add(LSTM(64, return_sequences=False))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(
    X_train_dl, y_train_dl,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_dl, y_test_dl)
)


In [ ]:
y_pred_dl = model.predict(X_test_dl)
y_pred_labels = y_pred_dl.argmax(axis=1)
y_true_labels = y_test_dl.argmax(axis=1)

print(classification_report(y_true_labels, y_pred_labels, target_names=le.classes_))
print("Akurasi =", accuracy_score(y_true_labels, y_pred_labels))
